## **Penggunaan Driver Neo4J**

Nama    : Kevin Davindio Atha

NIM     : 23031554173

Kelas   : Graf Sains Data 2023B 

#### **koneksi ke database Neo4J**

In [1]:
from neo4j import GraphDatabase, RoutingControl

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "admin123")

In [2]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Terhubung!")

Terhubung!


In [5]:
print(neo4j.__version__)

6.4.1


#### **Pembuatan sebuah Node berikut dengan relationship-nya**

In [3]:
def add_friend(driver, name, friend_name):
    driver.execute_query(
            "MERGE (a:Person {name: $name}) "
            "MERGE (friend:Person {name: $friend_name}) "
            "MERGE (a)-[:KNOWS]->(friend)",
            name=name, friend_name=friend_name, database_="neo4j"
        )

In [11]:
def print_friends(driver, name):
    records, _, _ = driver.execute_query(
    "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
    "RETURN friend.name ORDER BY friend.name", 
    name=name, database_="neo4j", routing_=RoutingControl.READ,
    )
    for record in records:
        print(record["friend.name"])
        
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Terhubung!")
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    add_friend(driver, "Guinevere", "Merlin")
    add_friend(driver, "Guinevere", "Arthur")
    print_friends(driver, "Arthur")
    print_friends(driver, "Guinevere")

Terhubung!
Guinevere
Lancelot
Merlin
Arthur
Merlin


In [12]:
def print_all_relations(driver):
    records, summary, keys = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(b:Person) "
        "RETURN a.name AS dari, b.name AS ke",
        database_="neo4j",
        routing_=RoutingControl.READ,
    )
    for record in records:
        print(f"{record['dari']} -> {record['ke']}")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print_all_relations(driver)

Arthur -> Guinevere
Arthur -> Lancelot
Arthur -> Merlin
Guinevere -> Arthur
Guinevere -> Merlin


### **Pertanyaan Pendahuluan**
1. Apa nama database yang digunakan pada tutorial ini?
2. Sebutkan akses lain (seperti sintaks DDL/DML CYPHER) yang dapat digunakan mengakses database lain dalam 1 instance!
3. Apa jenis graf pertemanan yang telah dibuat di sini?

### **Jawaban**
1. Nama database yang digunakan adalah neo4j. Hal ini terlihat pada parameter database_="neo4j"
2. Untuk mengakses database lain dalam satu instance Neo4j, Ada beberapa cara:
- Mengganti parameter `database_` pada `execute_query`, misalnya `database_="nama_database_lain"`.
- Memakai perintah Cypher `:use` di Neo4j Browser, contohnya `:use nama_database`.
- Memakai `USE` di dalam query Cypher, contohnya `USE nama_database MATCH (n) RETURN n`.
- Mengatur `database` pada session, misalnya `driver.session(database="nama_database")`.
- Mengelola database dengan perintah administrasi (dijalankan pada database `system`), seperti `CREATE DATABASE nama_database`, `SHOW DATABASES`, dan `DROP DATABASE nama_database`.
3. Graf pertemanan yang dibuat adalah graf berarah (directed graph). Alasannya, relasi dibuat dengan (a)-[:KNOWS]->(friend), yaitu ada tanda panah -> yang menunjukkan arah dari a ke friend. Artinya, Arthur “mengenal” Guinevere, Lancelot, dan Merlin, tetapi belum tentu sebaliknya.


### Impor Data dari RetailRocket ke Database Neo4J

In [1]:
import os
import sys
from neo4j import GraphDatabase
import argparse

uri = "bolt://127.0.0.1:7687"
user = "neo4j"
password = "admin123"   

driver = GraphDatabase.driver(uri, auth=(user, password))
driver.verify_connectivity()
print("Terhubung!")

session = driver.session(database="system")

Terhubung!


In [ ]:
import os
import pandas as pd
from getpass import getpass
from tqdm import tqdm
from neo4j import GraphDatabase

SOURCE = "ecommerce-dataset"          
csv_path = os.path.join(SOURCE, "events.csv")

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"'{csv_path}' tidak ditemukan. Folder kerja: {os.getcwd()}\n"
        f"Isi folder kerja: {os.listdir('.')}"
    )
print("events.csv ditemukan:", csv_path)

URI = "bolt://127.0.0.1:7687"
USER = "neo4j"
PASSWORD = "admin123"
DB_NAME = "neo4j"
BATCH_SIZE = 10000

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print("Terhubung!")

events.csv ditemukan: ecommerce-dataset\events.csv
Terhubung!


In [2]:
for q in [
    "CREATE CONSTRAINT visitor_id IF NOT EXISTS FOR (v:Visitor) REQUIRE v.id IS UNIQUE",
    "CREATE CONSTRAINT item_id IF NOT EXISTS FOR (i:Item) REQUIRE i.id IS UNIQUE",
    "CREATE CONSTRAINT category_id IF NOT EXISTS FOR (c:Category) REQUIRE c.id IS UNIQUE",
]:
    driver.execute_query(q, database_=DB_NAME)
print("Constraint siap.")

Constraint siap.


In [ ]:
QUERIES = {
    "view": """
        UNWIND $rows AS row
        MERGE (v:Visitor {id: row.visitorid})
        MERGE (i:Item {id: row.itemid})
        CREATE (v)-[:VIEW {timestamp: row.timestamp}]->(i)
    """,
    "addtocart": """
        UNWIND $rows AS row
        MERGE (v:Visitor {id: row.visitorid})
        MERGE (i:Item {id: row.itemid})
        CREATE (v)-[:ADDTOCART {timestamp: row.timestamp}]->(i)
    """,
    "transaction": """
        UNWIND $rows AS row
        MERGE (v:Visitor {id: row.visitorid})
        MERGE (i:Item {id: row.itemid})
        CREATE (v)-[:TRANSACTION {timestamp: row.timestamp,
        transactionid: row.transactionid}]->(i)
    """,
}

total_rows = sum(1 for _ in open(csv_path, encoding="utf-8")) - 1
total_chunks = total_rows // BATCH_SIZE + 1

with driver.session(database=DB_NAME) as session:
    for chunk in tqdm(pd.read_csv(csv_path, chunksize=BATCH_SIZE),
        total=total_chunks, desc="Impor events"):
        chunk["transactionid"] = chunk["transactionid"].astype("Int64")
        chunk = chunk.astype(object).where(chunk.notnull(), None)
        for event, query in QUERIES.items():
            rows = chunk[chunk["event"] == event].to_dict("records")
            if rows:
                session.execute_write(lambda tx, q=query, r=rows: tx.run(q, rows=r).consume())
print("Events selesai.")

Impor events: 100%|██████████| 276/276 [07:14<00:00,  1.57s/it]

Events selesai.
